In [1]:
import httpx
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time
import json
import psycopg
import os
import sys
import numpy as np
from dotenv import load_dotenv
from itertools import zip_longest

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
api_key = os.getenv("API_KEY")
api_url = 'https://api.stratz.com/graphql'
headers = {
    'User-Agent': 'STRATZ_API',
    "Authorization": f"Bearer {api_key}"
}

In [8]:
patches_response = httpx.get(f'{api_url}/constants/patch').json()

In [13]:
patches_response

[{'name': '6.70', 'date': '2010-12-24T00:00:00Z', 'id': 0},
 {'name': '6.71', 'date': '2011-01-21T00:00:00Z', 'id': 1},
 {'name': '6.72', 'date': '2011-04-27T00:00:00Z', 'id': 2},
 {'name': '6.73', 'date': '2011-12-24T00:00:00Z', 'id': 3},
 {'name': '6.74', 'date': '2012-03-10T00:00:00Z', 'id': 4},
 {'name': '6.75', 'date': '2012-09-30T00:00:00Z', 'id': 5},
 {'name': '6.76', 'date': '2012-10-21T00:00:00Z', 'id': 6},
 {'name': '6.77', 'date': '2012-12-15T00:00:00Z', 'id': 7},
 {'name': '6.78', 'date': '2013-05-30T00:00:00Z', 'id': 8},
 {'name': '6.79', 'date': '2013-11-24T00:00:00Z', 'id': 9},
 {'name': '6.80', 'date': '2014-01-27T00:00:00Z', 'id': 10},
 {'name': '6.81', 'date': '2014-04-29T00:00:00Z', 'id': 11},
 {'name': '6.82', 'date': '2014-09-24T00:00:00Z', 'id': 12},
 {'name': '6.83', 'date': '2014-12-17T00:00:00Z', 'id': 13},
 {'name': '6.84', 'date': '2015-04-30T21:00:00Z', 'id': 14},
 {'name': '6.85', 'date': '2015-09-24T20:00:00Z', 'id': 15},
 {'name': '6.86', 'date': '2015-12

In [10]:
patches_df = pd.DataFrame(patches_response)

In [16]:
patches_df['date'] = pd.to_datetime(patches_df['date'], format='ISO8601')

In [18]:
patches_df.dtypes

name                 object
date    datetime64[ns, UTC]
id                    int64
dtype: object

In [19]:
dbf.create_table_from_df(patches_df, 'patches', conn_str)

Table 'patches' created successfully.


In [20]:
dbf.insert_df_into_table(patches_df, 'patches', conn_str)

Data inserted into table 'patches' successfully.


In [2]:
heroes_response = httpx.get(f'{api_url}/constants/heroes').json()

In [29]:
heroes_df = pd.DataFrame(heroes_response[key] for key in heroes_response.keys())

In [30]:
heroes_df = heroes_df.drop('img', axis=1)
heroes_df = heroes_df.drop('icon', axis=1)
heroes_df = heroes_df.drop('legs', axis=1)

In [33]:
dbf.create_table_from_df(heroes_df, 'heroes', conn_str)

Table 'heroes' created successfully.


In [34]:
dbf.insert_df_into_table(heroes_df, 'heroes', conn_str)

Data inserted into table 'heroes' successfully.


In [ ]:
# Get game versions
query = """
    query {
        constants {
            gameVersions {
                id
                name
                asOfDateTime
            }
        }
    }
"""
result = dbf.query_stratz(query, headers=headers, api_url=api_url)
df = pd.DataFrame(result['data']['constants']['gameVersions'])
df['asOfDateTime'] = df['asOfDateTime'].apply(datetime.fromtimestamp)
dbf.create_table_from_df(df, 'patches', conn_str)
dbf.insert_df_into_table(df, 'patches', conn_str)

In [ ]:
## Get npc data from stratz
query = """
    query($gameVersionId: Short!) {
        constants {
            npcs(gameVersionId: $gameVersionId) {
                id
                name
                stat {
                    statusHealth
                    statusHealthRegen
                    attackDamageMin
                    attackDamageMax
                    attackRate
                    attackRange
                    movementSpeed
                    isNeutralUnitType
                    isAncient
                    teamName
                }
            }
        }
    }
"""
variables = {'gameVersionId': 182} #TODO: replace hardcoded value
result = dbf.query_stratz(query, headers=headers, api_url=api_url, variables=variables)
df = pd.DataFrame(result['data']['constants']['npcs'])
stats_df = pd.json_normalize(df['stat'])
df = df.join(stats_df).drop(columns=['stat'])
discard_patterns = [
    'thinker', 'companion', 'visual', 'sound', 'event', 
    'shmup', 'banana', 'target_dummy', 'looping', 'promo'
]
discard_regex = '|'.join(discard_patterns)
df_filtered = df[~df['name'].str.contains(discard_regex, case=False, na=False)]
df_filtered.dtypes
df_filtered.convert_dtypes(convert_integer=False).dtypes
dbf.create_table_from_df(df_filtered, 'npcs', conn_str, False)
dbf.insert_df_into_table(df_filtered, 'npcs', conn_str)

In [2]:
query = """
    query($gameVersionId: Short!) {
        constants {
            items(gameVersionId: $gameVersionId) {
                id
                        shortName
                displayName
                isSupportFullItem
                attributes {
                    name
                    value
                }
                stat {
                    cost
                    isRecipe
                    isSupport
                    behavior
                    manaCost
                    shopTags
                    needsComponents
                    itemResult
                    quality
                }
                components {
                    componentId
                }
            }
        }
    }
"""
variables = {'gameVersionId': 182} #TODO: replace hardcoded value
result = dbf.query_stratz(query, headers=headers, api_url=api_url, variables=variables)

In [9]:
result_json = result['data']['constants']['items']
df = pd.json_normalize(result_json)
df = df.drop(['stat'], axis=1)
rename_dict = {}
for col_name in df.columns:
    if col_name.startswith('stat.'):
        rename_dict[col_name] = col_name[5:]
df = df.rename(rename_dict, axis=1)
## I used non-na components because isRecipe includes old
## recipes for which items are not active in current patch
df_recipes = df[df['components'].notna()]
df['behavior'] = df['behavior'].astype('Int64')
df['cost'] = df['cost'].astype('Int64')
df['itemResult'] = df['itemResult'].astype('Int64')

In [ ]:
#TODO Database design:
## Table 1 - basic item details
## Table 3 - item recipes

In [10]:
ids = []
shop_tags = []
for idx, row in df.iterrows():
    ids.append(row['id'])
    try:
        shop_tags.append(row['shopTags'].split(';'))
    except:
        shop_tags.append(row['shopTags'])
df = df.drop('shopTags', axis=1)
df_attributes_jsonb = df.copy()
for idx, row in df.copy().iterrows():
    flat_attrs = {}
    if isinstance(row['attributes'], list):
        for attr in row['attributes']:
            flat_attrs[attr['name']] = attr['value']
            df_attributes_jsonb.at[idx, 'attributes'] = flat_attrs
    else:
        df_attributes_jsonb.at[idx, 'attributes'] = pd.NA

In [11]:
rename_dict = {}
for col_name in df_attributes_jsonb.columns:
    new_colname = col_name
    for idx, char in enumerate(col_name):
        if str.isupper(char):
            new_colname = new_colname[: idx] + '_' + new_colname[idx:]
    new_colname = new_colname.lower()
    rename_dict[col_name] = new_colname
rename_dict['isSupportFullItem'] = 'is_support_full_item'
df_attributes_jsonb.rename(rename_dict, axis=1)

,id,short_name,display_name,is_support_full_item,attributes,components,cost,is_recipe,is_support,behavior,mana_cost,needs_components,item_result,quality
0,1,blink,Blink Dagger,None,"{'blink_damage_cooldown': '3.0', 'blink_range'...",None,2250,False,False,137439478800,[0],False,<NA>,component
1,2,blades_of_attack,Blades of Attack,None,{'bonus_damage': '9'},None,450,False,False,2,None,False,<NA>,component
2,3,broadsword,Broadsword,None,{'bonus_damage': '15'},None,1000,False,False,2,None,False,<NA>,component
3,4,chainmail,Chainmail,None,{'bonus_armor': '4'},None,550,False,False,2,None,False,<NA>,component
4,5,claymore,Claymore,None,{'bonus_damage': '20'},None,1350,False,False,2,None,False,<NA>,component
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,4207,recipe_great_famango,,None,<NA>,"[{'componentId': 4204}, {'componentId': 4204},...",0,True,False,0,None,False,4205,None
571,4208,recipe_greater_famango,,None,<NA>,"[{'componentId': 4205}, {'componentId': 4205}]",0,True,False,0,None,False,4206,None
572,4300,ofrenda,Beloved Memory,None,{'speed': '1000'},None,0,False,False,72,[0],False,<NA>,None
573,4301,ofrenda_shovel,Scrying Shovel,None,<NA>,None,0,False,False,134217936,[0],False,<NA>,None


In [12]:
df_attributes_jsonb.dtypes

id                    int64
shortName            object
displayName          object
isSupportFullItem    object
attributes           object
components           object
cost                  Int64
isRecipe             object
isSupport            object
behavior              Int64
manaCost             object
needsComponents      object
itemResult            Int64
quality              object
dtype: object

In [21]:
dbf.create_table_from_df(df_attributes_jsonb, 'item_details', conn_str, convert_dtypes=False, jsonb_cols=['attributes', 'components'])

Table 'item_details' created successfully.


In [19]:
mask = df_attributes_jsonb.apply(lambda col: col.astype(str).str.contains('\x00', na=False))
rows_with_nul = df_attributes_jsonb[mask.any(axis=1)]
print(rows_with_nul)

Empty DataFrame
Columns: [id, shortName, displayName, isSupportFullItem, attributes, components, cost, isRecipe, isSupport, behavior, manaCost, needsComponents, itemResult, quality]
Index: []


In [18]:
## Since these are not active items currently (one is no longer, other not yet possibly)
## rows are simply dropped
df_attributes_jsonb = df_attributes_jsonb.drop([416, 430], axis=0)

In [22]:
dbf.insert_df_into_table(df_attributes_jsonb, 'item_details', conn_str, jsonb_cols=['attributes', 'components'])

Data inserted into table 'item_details' successfully.


In [ ]:
query = '''
    query($gameVersionId: Short) {
        constants {
            heroes(gameVersionId: $gameVersionId) {
                id
                name
                displayName
                shortName
                gameVersionId
                roles {
                    roleId
                    level
                }
                abilities {
                    slot
                    gameVersionId
                    abilityId
                }
                stats {
                    startingArmor
                    startingMagicArmor
                    startingDamageMin
                    startingDamageMax
                    attackRange
                    attackType
                    moveSpeed
                    hpRegen
                    mpRegen
                    primaryAttribute
                    strengthGain
                    agilityGain
                    intelligenceGain
                    strengthBase
                    agilityBase
                    intelligenceBase
                }
                talents {
                    abilityId
                    slot
                }
                facets {
                    abilityId
                    facetId
                    slot
                }
            }
        }
    }
'''
result = dbf.query_stratz(query, headers, api_url, variables={'gameVersionId': 182})
res = result['data']['constants']['heroes']

In [ ]:
df_hero_details = pd.DataFrame(res)
df_hero_abilities = pd.DataFrame(
    columns=[
        'heroId',
        'slot',
        'gameVersionId',
        'abilityId'
    ]
)
df_hero_talents = pd.DataFrame(
    columns=[
        'heroId',
        'abilityId',
        'slot'
    ]
)
df_hero_facets = pd.DataFrame(
    columns=[
        'heroId',
        'abilityId',
        'facetId',
        'slot'
    ]
)
for idx, row in df_hero_details.iterrows():
    df_ha = pd.DataFrame(row['abilities'])
    df_ha.insert(0, 'heroId', row['id'])
    df_hero_abilities = pd.concat([df_hero_abilities, df_ha])
    df_ht = pd.DataFrame(row['talents'])
    df_ht.insert(0, 'heroId', row['id'])
    df_hero_talents = pd.concat([df_hero_talents, df_ht])
    df_hf = pd.DataFrame(row['facets'])
    df_hf.insert(0, 'heroId', row['id'])
    df_hero_facets = pd.concat([df_hero_facets, df_hf])
df_hero_details = df_hero_details.drop(['abilities', 'talents', 'facets'], axis=1)
dbf.create_table_from_df(df_hero_details, 'hero_details', conn_str, convert_dtypes=False, jsonb_cols=['roles', 'stats'])
dbf.insert_df_into_table(df_hero_details, 'hero_details', conn_str, jsonb_cols=['roles', 'stats'])
dbf.create_table_from_df(df_hero_abilities, 'hero_abilities', conn_str, add_serial_id=True)
dbf.insert_df_into_table(df_hero_abilities, 'hero_abilities', conn_str)
dbf.create_table_from_df(df_hero_talents, 'hero_talents', conn_str, add_serial_id=True)
dbf.insert_df_into_table(df_hero_talents, 'hero_talents', conn_str)
dbf.create_table_from_df(df_hero_facets, 'hero_facets', conn_str, add_serial_id=True)
dbf.insert_df_into_table(df_hero_facets, 'hero_facets', conn_str)

C:\Users\benib\AppData\Local\Temp\ipykernel_50952\76676605.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_hero_facets = pd.concat([df_hero_facets, df_hf])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\76676605.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_hero_facets = pd.concat([df_hero_facets, df_hf])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\76676605.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a fu

Table 'hero_details' created successfully.
Data inserted into table 'hero_details' successfully.
Table 'hero_abilities' created successfully.
Data inserted into table 'hero_abilities' successfully.
Table 'hero_talents' created successfully.
Data inserted into table 'hero_talents' successfully.
Table 'hero_facets' created successfully.
Data inserted into table 'hero_facets' successfully.


In [48]:
query = '''
    query {
        constants {
            abilities(gameVersionId: 182) {
            id
            name
            uri
            language {
                displayName
                description
                aghanimDescription
                shardDescription
            }
            stat {
                abilityId
                type
                behavior
                unitDamageType
                unitTargetType
                unitTargetTeam
                unitTargetFlags
                duration
                damage
                castPoint
                castRange
                channelTime
                manaCost
                cooldown
                isGrantedByScepter
                isGrantedByShard
                hasScepterUpgrade
                hasShardUpgrade
                dispellable
                isInnate
                isUltimate
                linkedAbilityId
            }
            attributes {
                name
                value
                linkedSpecialBonusAbilityId
                requiresScepter
            }
            isTalent
        }
    }
}
'''
result = dbf.query_stratz(query, headers, api_url)
res = result['data']['constants']['abilities']

In [113]:
def clean_lists(val):
    if isinstance(val, list):
        return ", ".join(map(str, val))
    return val

df_abilities = pd.DataFrame(res)
abilities_language = pd.json_normalize(df_abilities['language'])
new_cols = ['ability_' + colname for colname in abilities_language.columns]
rename_dict = {old_col: new_col for old_col, new_col in zip(abilities_language.columns, new_cols)}
abilities_language = abilities_language.rename(rename_dict, axis=1)
df_abilities = pd.concat([df_abilities, abilities_language], axis=1).drop('language', axis=1)
ability_stats = pd.json_normalize(df_abilities['stat']).drop('abilityId', axis=1)
df_abilities = pd.concat([df_abilities, ability_stats], axis=1).drop(['stat', 'attributes'], axis=1)
list_cols = ['duration', 'damage', 'castPoint', 'castRange', 'channelTime', 'manaCost', 'cooldown']
for col in list_cols:
    df_abilities[col] = df_abilities[col].apply(clean_lists)
dbf.create_table_from_df(df_abilities, 'ability_details', conn_str)
dbf.insert_df_into_table(df_abilities, 'ability_details', conn_str)

Table 'ability_details' created successfully.
Data inserted into table 'ability_details' successfully.


In [103]:
df_abilities

,id,name,uri,isTalent,ability_displayName,ability_description,ability_aghanimDescription,ability_shardDescription,type,behavior,...,manaCost,cooldown,isGrantedByScepter,isGrantedByShard,hasScepterUpgrade,hasShardUpgrade,dispellable,isInnate,isUltimate,linkedAbilityId
0,82,bear_empty1,None,False,None,[],None,None,0.0,6.600000e+01,...,None,None,False,False,False,False,NONE,False,False,NaN
1,83,bear_empty2,None,False,None,[],None,None,0.0,6.600000e+01,...,None,None,False,False,False,False,NONE,False,False,NaN
2,84,courier_autodeliver,None,False,Auto Deliver,[Toggle auto delivering of items.],None,None,0.0,6.885370e+10,...,None,None,False,False,False,False,NONE,False,False,NaN
3,195,ringmaster_spotlight,None,False,Spotlight,[Ringmaster shines 3 spotlights that sweep ove...,None,None,0.0,4.800000e+01,...,[50],[30],False,True,False,False,NO,False,False,NaN
4,196,ringmaster_summon_unicycle,None,False,Unicycle,[<h1>Use: Saddle Up</h1>Summons a speedy unicy...,None,None,0.0,7.045464e+13,...,None,[3],False,False,False,False,NONE,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2973,9998,roshan_halloween_apocalypse,None,False,None,[],None,None,0.0,4.000000e+00,...,[0],[20],False,False,False,False,NONE,False,False,NaN
2974,9999,roshan_halloween_burn,None,False,None,[],None,None,0.0,4.800000e+01,...,[0],[30],False,False,False,False,NONE,False,False,NaN
2975,10000,roshan_halloween_levels,None,False,None,[],None,None,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2976,10001,roshan_halloween_summon,None,False,None,[],None,None,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [91]:
df_ability_attributes = pd.DataFrame(
    columns=[
        'abilityId',
        'name',
        'value',
        'linkedSpecialBonusAbilityId',
        'requiresScepter'
    ]
)
for data in res:
    df_aa = pd.DataFrame(data['attributes'])
    df_aa.insert(0, 'abilityId', data['id'])
    df_ability_attributes = pd.concat([df_ability_attributes, df_aa])

C:\Users\benib\AppData\Local\Temp\ipykernel_50952\971139022.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_ability_attributes = pd.concat([df_ability_attributes, df_aa])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\971139022.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_ability_attributes = pd.concat([df_ability_attributes, df_aa])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\971139022.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA

In [92]:
df_ability_attributes

,abilityId,name,value,linkedSpecialBonusAbilityId,requiresScepter
0,195,duration,8,NaN,False
1,195,illusion_percent_damage,30,NaN,False
2,195,initial_speed,3000,NaN,False
3,195,linger_duration,0.3,NaN,False
4,195,miss_chance,30,NaN,False
...,...,...,...,...,...
0,9999,damage,1000,NaN,False
1,9999,projectile_count,20,NaN,False
2,9999,radius,200,NaN,False
3,9999,rotation_angle,90,NaN,False
